[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q4/03_per_disease_interpretation.ipynb)

# R2-Q4 Week 4 — Interpret transfer outcomes per disease

### This notebook produces `interpretation_table.parquet`, the headline finding for your Week 4 paper and final presentation.

This notebook takes the per-disease gap measurements from Week 3 and pairs each with a transfer-outcome interpretation. The question is not just "did transfer hold or collapse" but *what that means*: a small gap can be a model that learned the disease or one that learned generic leaf appearance; a large gap can be a host-template shortcut or two genuinely different diseases sharing one name. Telling those apart needs a second fact — whether the same pathogen causes the disease on both hosts — that the gap alone can't give you. Where the page can't settle that fact, "unclear" is a legitimate, reportable outcome. Per-disease claims, not one sweeping statement, are what your paper-level conclusion should commit to.


By the end of this notebook you will have:

- A per-disease interpretation table saved as `interpretation_table.parquet`, placing each disease on the page's 2×2 grid (transfer outcome × pathogen identity).
- A brief Noyan (2022) sanity check noting where the background-shortcut result constrains your interpretation.
- A short paper-level conclusion drafted per-disease (per Consideration 4 — claims your data supports, not sweeping ones).
- Submitted final presentation and final paper.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R2-Q4 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r2_q4")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Defensive load of `precommit.json`, the trained classifier, and the gap table

Week 3 measured a gap for each disease: how far its accuracy fell when the
classifier was asked to recognize it on a host it had never trained on. This
notebook reads those gaps and turns each one into an *interpretation* — a claim
about what the model learned. That is the whole job of Week 4, and it is a
different kind of work from the three weeks before it. There is almost no new
computation here. The output is a table of readings, and a paragraph of
conclusion, that only you can write.

The load is defensive, the same way every notebook in this chain has been: a
missing or half-written upstream file should fail right here, with a message
that tells you which notebook to re-run, rather than partway through with a
confusing error. You read three things — the Week-1 precommit (to recompose the
label space and find the transfer subjects), and the two files NB02 wrote: the
gap table (the per-disease numbers you interpret) and the comparison summary
(which carries the label order you assert against).

In [ ]:
# Read back the Week-1 decisions and recompose the label space exactly as NB01
# and NB02 did. The label space is every disease the classifier predicts; the
# transfer subjects are the subset with a withheld host — the only diseases with
# a cross-host gap to interpret. This is the same composition NB02 ran; it has
# to agree with the label order NB02 saved, which the next cell asserts.
import json

precommit_path = OUTPUT_DIR / "precommit.json"
if not precommit_path.exists():
    raise FileNotFoundError(
        f"No precommit.json at {precommit_path}. Run NB00 "
        "(00_orient_and_precommit.ipynb) to completion first — it writes the "
        "Week-1 decisions this notebook reads."
    )
with open(precommit_path) as f:
    precommit = json.load(f)

for key in ("disease_list", "hold_out_scheme", "cross_host_aggregation"):
    if precommit.get(key) is None:
        raise KeyError(
            f"precommit.json is missing '{key}'. Re-run NB00 — every decision "
            "should be filled in before you reach Week 4."
        )

# Compose the label space: multi-host diseases, then any single-host controls,
# then "healthy" if it was included. The ORDER defines the classifier's
# output-index space and must match what NB01 trained and NB02 saved.
dl = precommit["disease_list"]
diseases = list(dl["multi_host_diseases"]) + list(dl["control_single_host_diseases"])
if dl.get("include_healthy"):
    diseases = diseases + ["healthy"]

held_out = precommit["hold_out_scheme"]["held_out_host"]
num_classes = len(diseases)

# Chance level and the near-chance floor, defined exactly as NB01 did: chance is
# 1/N for an N-disease classifier; "twice chance" (2/N) is the floor below which
# an accuracy is treated as barely-better-than-guessing. Section 2 uses these to
# decide a disease's transfer-outcome row — so it reuses NB01's anchor rather
# than inventing a new threshold.
chance_level = 1 / num_classes
near_chance_floor = 2 / num_classes

# Transfer subjects: the diseases with a withheld host. These, and only these,
# get a row in the gap table and a cell on the 2×2 grid.
transfer_diseases = [d for d in diseases if d in held_out]

print(f"Label space ({num_classes}): {', '.join(diseases)}")
print(f"Chance level: {chance_level:.3f}   Near-chance floor (2x chance): "
      f"{near_chance_floor:.3f}")
print(f"Transfer subjects ({len(transfer_diseases)}): "
      f"{', '.join(transfer_diseases)}")

In [ ]:
import pandas as pd

# 1. The gap table NB02 wrote — one row per transfer subject (plus an optional
#    "overall" row if you aggregated that way), with the within-host accuracy,
#    the cross-host accuracy, the gap, the cross-host sample count, and the
#    near-chance flag NB01 set, carried forward.
gap_table_path = OUTPUT_DIR / "gap_table.parquet"
if not gap_table_path.exists():
    raise FileNotFoundError(
        f"No gap_table.parquet at {gap_table_path}. Run NB02 "
        "(02_cross_host_evaluation.ipynb) to completion first — it writes the "
        "per-disease gaps this notebook interprets."
    )
gap_table = pd.read_parquet(gap_table_path)

# Assert the columns are exactly what NB02 wrote, rather than reaching for a
# column by name later and getting a confusing KeyError deep in Section 2.
expected_cols = {
    "disease", "within_host_acc", "cross_host_acc", "gap",
    "n_cross_host", "within_host_flag",
}
missing = expected_cols - set(gap_table.columns)
if missing:
    raise ValueError(
        f"gap_table.parquet is missing columns {sorted(missing)}. Re-run NB02 — "
        "this notebook's interpretation is built on its exact output schema."
    )

# 2. NB02's comparison summary — carries the label order (assert on it), the
#    transfer-subject list, and the headline numbers you read into the
#    conclusion. The label-order assert is the same spine NB02 had: if the order
#    drifted, an output index means a different disease here than in training and
#    every reading below is silently wrong.
summary_path = OUTPUT_DIR / "comparison_summary.json"
if not summary_path.exists():
    raise FileNotFoundError(
        f"No comparison_summary.json at {summary_path}. Re-run NB02 — it writes "
        "the record this notebook reads the label order back from."
    )
with open(summary_path) as f:
    comparison_summary = json.load(f)

saved_diseases = comparison_summary["methodology"]["diseases"]
assert diseases == saved_diseases, (
    "Label-space mismatch between this notebook and NB02.\n"
    f"  NB03 composed from precommit: {diseases}\n"
    f"  NB02 saved:                   {saved_diseases}\n"
    "These must be identical — same diseases, same order. If you changed the "
    "disease list since NB02 ran, re-run NB00, NB01, and NB02 before this one."
)

# A per-disease view of the gap table, indexed by disease, for Section 2. Drop
# the optional "overall" row — Section 2 interprets diseases, not the aggregate.
per_disease = gap_table[gap_table["disease"] != "overall"].set_index("disease")

print(f"Loaded gap_table.parquet ({len(per_disease)} per-disease row(s))")
print("Label order confirmed against NB02.")
print(f"Overall gap (within - cross): "
      f"{comparison_summary['results']['overall_gap']:.3f}")
flagged = comparison_summary["results"].get("flagged_diseases", [])
if flagged:
    print(f"Near-chance flags carried from NB01: {', '.join(flagged)}")
else:
    print("No near-chance flags carried forward.")

### 2) Per-disease interpretation against the 2×2 grid (transfer outcome × pathogen identity)

This is the heart of the notebook. For each transfer subject you place its
result on a 2×2 grid, then read off what that placement means about the model.
The grid has two axes, and they come from two different places.

**The first axis — transfer outcome — you measured.** It is in the gap table:
did the disease's accuracy hold up on the withheld host, or collapse toward
chance? You will compute this row in the practice below, straight from the
cross-host accuracy.

**The second axis — pathogen identity — you have to supply.** For each disease,
is it caused by the *same pathogen* on both hosts, or by *different pathogens*
that happen to share a common name? The gap table cannot tell you this; it is a
fact about the biology, not the data, and it comes from the R2-Q4 question page.
You fill it in, with a one-line reason, in the fill cell below.

The two axes together separate four cases the gap alone cannot:

|                              | **Same pathogen**                 | **Different pathogens (shared name)** |
|------------------------------|-----------------------------------|---------------------------------------|
| **Transfer held (gap small)**| Clean transfer — learned the disease | Learned generic leaf appearance     |
| **Transfer collapsed (gap large)** | Host-template shortcut       | Two genuinely different diseases under one label |

A disease where you cannot establish pathogen identity from the page does not
get forced into a column — it lands in an honest `unclear`, which is itself a
finding (you measured the transfer but cannot yet say *why*). And a disease NB01
flagged near-chance even within host is `unclear` for a different reason: there
was no within-host skill to transfer, so its gap, large or small, does not
support a clean reading either way.

**On the transfer-outcome row, you reuse the chance anchor from Week 2, not a
new threshold.** NB01 flagged a disease near-chance when its accuracy fell to or
below twice chance (`2/N`). The same logic decides the row here: a cross-host
accuracy that stays clearly above that floor is "transfer held"; one that falls
to or below it has collapsed toward guessing. Tying the row to distance-from-
chance, rather than to an invented gap cutoff, keeps it continuous with the
reasoning you already did.

### Practice 2.1 — Compute each disease's transfer-outcome row

Fill in the body of `transfer_row` below. Given a disease's cross-host accuracy
and its within-host flag, return one of three labels:

- `"held"` — the cross-host accuracy is clearly above the near-chance floor;
  transfer held up.
- `"collapsed"` — the cross-host accuracy is at or below the near-chance floor;
  transfer collapsed toward guessing.
- `"uninterpretable"` — the disease was flagged near-chance *within* host
  (`within_host_flag == "near-chance"`); there was little skill to transfer, so
  the gap does not support a clean read regardless of its size. Check this
  *first* — it takes priority over the cross-host number.

The floor is `near_chance_floor` from Section 1 (`2/num_classes`). This is the
one piece of the placement that is mechanical — the column and the reasoning in
the next cell are yours.

In [ ]:
def transfer_row(cross_host_acc: float, within_host_flag: str) -> str:
    """Return the transfer-outcome row for one disease.

    Parameters
    ----------
    cross_host_acc : the disease's cross-host accuracy (from the gap table).
    within_host_flag : "ok" or "near-chance", carried from NB01.

    Returns one of: "held", "collapsed", "uninterpretable".
    """
    # HINT: three branches.
    #  1. If the within-host flag is "near-chance", return "uninterpretable"
    #     FIRST — a disease with no within-host skill can't show real transfer.
    #  2. Otherwise compare cross_host_acc to near_chance_floor (Section 1):
    #     above the floor -> "held"; at or below -> "collapsed".
    raise NotImplementedError(
        "Fill in transfer_row: check the near-chance flag first, then compare "
        "cross_host_acc to near_chance_floor."
    )


# Apply it to every transfer subject, reading each disease's numbers off the
# per-disease gap table from Section 1.
transfer_outcome = {}
for d in transfer_diseases:
    transfer_outcome[d] = transfer_row(
        cross_host_acc=float(per_disease.loc[d, "cross_host_acc"]),
        within_host_flag=per_disease.loc[d, "within_host_flag"],
    )

print(f"{'disease':18s} {'cross-host acc':>14s} {'within flag':>14s} {'row':>16s}")
for d in transfer_diseases:
    print(f"{d:18s} {per_disease.loc[d, 'cross_host_acc']:>14.3f} "
          f"{per_disease.loc[d, 'within_host_flag']:>14s} "
          f"{transfer_outcome[d]:>16s}")

Now the axis only you can supply. For each transfer subject, fill in two things
in `pathogen_identity` below:

- `pathogen` — one of `"same"`, `"different"`, or `"unclear"`:
  - `"same"` — the same pathogen species causes this disease on both the
    training hosts and the withheld host.
  - `"different"` — the disease name covers *different* pathogens on different
    hosts (a shared common name, distinct organisms).
  - `"unclear"` — the question page does not let you establish which. This is a
    legitimate answer, not a cop-out; an honest `unclear` is better than a
    confident guess.
- `reason` — one sentence, in your own words, citing what on the R2-Q4 question
  page supports your call. This is the sentence that makes the placement a claim
  rather than an assertion.

Every transfer subject needs an entry. The cell after this one checks that you
filled each `reason` with something real, not the placeholder — and refuses to
build the grid until you have.

In [ ]:
# EDIT THIS CELL. One entry per transfer subject. The diseases are listed for
# you from Section 1; fill the "pathogen" and "reason" for each. Leave the keys
# as they are — the validation below checks every transfer subject is covered.
pathogen_identity = {
    # Example of the shape (not an answer — replace with your own calls):
    #   "Early_blight": {
    #       "pathogen": "same",
    #       "reason": "PLACEHOLDER — REWRITE: cite the page's pathogen note.",
    #   },
}

# Seed any missing diseases with placeholders so the dict's shape is obvious and
# the validation below names exactly what's left to fill. (This adds entries; it
# does not overwrite anything you wrote.)
for d in transfer_diseases:
    pathogen_identity.setdefault(
        d, {"pathogen": "unclear", "reason": "PLACEHOLDER — REWRITE BEFORE SUBMISSION."}
    )

print("Fill the pathogen identity and a one-line reason for each disease above,")
print("then run the next cell to validate and build the grid.")
for d in transfer_diseases:
    e = pathogen_identity[d]
    print(f"  {d:18s} pathogen={e['pathogen']:>9s}  reason={e['reason'][:48]}")

In [ ]:
# Two-part check, the same structural-then-semantic pattern the gates use.
# Structural: every transfer subject present, every "pathogen" a valid value.
# Semantic: every "reason" actually written — non-empty and not the placeholder.
VALID_PATHOGEN = {"same", "different", "unclear"}

problems = []
for d in transfer_diseases:
    if d not in pathogen_identity:
        problems.append(f"  {d}: no entry")
        continue
    e = pathogen_identity[d]
    if e.get("pathogen") not in VALID_PATHOGEN:
        problems.append(
            f"  {d}: pathogen={e.get('pathogen')!r} is not one of {sorted(VALID_PATHOGEN)}"
        )
    reason = (e.get("reason") or "").strip()
    if not reason or reason.startswith("PLACEHOLDER"):
        problems.append(f"  {d}: reason is empty or still the placeholder")

if problems:
    raise ValueError(
        "The pathogen-identity fill isn't complete:\n"
        + "\n".join(problems)
        + "\n\nEdit pathogen_identity in the cell above — every transfer subject "
        "needs a valid pathogen value and a real one-line reason — then re-run."
    )

# Both axes are in hand: combine them into the 2×2 cell name. A disease lands in
# "unclear" if EITHER axis is unclear — an uninterpretable row (no within-host
# skill) or an unclear pathogen column. The four named cells follow the grid in
# Section 2.a.
def quadrant(row: str, pathogen: str) -> str:
    if row == "uninterpretable" or pathogen == "unclear":
        return "unclear"
    if row == "held" and pathogen == "same":
        return "clean_transfer"
    if row == "held" and pathogen == "different":
        return "generic_leaf_appearance"
    if row == "collapsed" and pathogen == "same":
        return "host_template_shortcut"
    if row == "collapsed" and pathogen == "different":
        return "different_diseases_one_label"
    return "unclear"  # any combination not covered above

placement = {}
for d in transfer_diseases:
    row = transfer_outcome[d]
    pathogen = pathogen_identity[d]["pathogen"]
    placement[d] = {
        "row": row,
        "pathogen": pathogen,
        "quadrant": quadrant(row, pathogen),
        "reason": pathogen_identity[d]["reason"].strip(),
    }

print(f"{'disease':18s} {'row':>16s} {'pathogen':>10s} {'quadrant':>28s}")
for d in transfer_diseases:
    p = placement[d]
    print(f"{d:18s} {p['row']:>16s} {p['pathogen']:>10s} {p['quadrant']:>28s}")

### 3) Background-shortcut sanity check: revisit Noyan (2022)

Before you write the interpretation table, one finding has to sit alongside it,
because it limits what any of your placements can claim. Noyan (2022) showed
that PlantVillage's disease labels can be predicted from the image *background*
alone: with the background downsampled to just eight pixels — no leaf at all — a
classifier reached about 49% accuracy on the 38-class task, against roughly 3%
for guessing. The lab-capture conventions that make PlantVillage clean (uniform
backdrops, controlled lighting, one leaf per shot) also tie the backdrop to the
label, and a model can score well by reading the backdrop instead of the leaf.

Worse for a transfer study, the bias is not confined to the background. Noyan's
Table 6 found that even foreground-only imagery, blurred to remove fine detail,
still predicted the class well above chance — the capture bias has leaked into
the leaf region itself. So there is no clean "background versus leaf" line you
can draw and declare yourself safe on one side of it.

Hold this against your 2×2 grid. The shortcut is not a separate worry bolted on
afterward — it is already *named* in two of your four cells:

- **`host_template_shortcut`** (collapsed transfer, same pathogen) is the
  shortcut showing up directly. If the same pathogen causes a disease on both
  hosts and accuracy still collapses on the withheld host, a leading explanation
  is that the model keyed on host-specific appearance — the kind of backdrop-
  and-capture regularity Noyan measured — rather than on the lesion that the two
  hosts share.
- **`generic_leaf_appearance`** (held transfer, different pathogens) is the same
  hazard wearing the opposite face: accuracy that *holds* across hosts for what
  are really two different diseases suggests the model is leaning on something
  common to both images that isn't the pathogen — again, plausibly the shared
  capture style.

The constraint this puts on your readings is specific: a `clean_transfer`
placement is the *only* one of the four that claims the model learned genuine
disease features, and Noyan's foreground result means you cannot fully clear
even that cell. The honest scope of a clean-transfer claim is "accuracy held
across hosts for the same pathogen, consistent with the model having learned
disease features" — not "the model learned disease features." This notebook does
not have the attention maps that would let you check directly where the model
looked (that is R2-Q2's Grad-CAM work, on a different chain). So the move here is
to carry the caveat in words, into the conclusion you write in Section 5, rather
than to claim a cleanliness you cannot demonstrate.

### 4) Write the per-disease interpretation table

You have, for each transfer subject, its placement on the grid and a one-line
reason. Now join that to the numbers behind it — the within-host and cross-host
accuracy and the gap — into a single table. This is the artifact the notebook
exists to produce: each row is one disease's transfer outcome, its 2×2 cell, and
the reason you assigned it. The conclusion in Section 5 is written *from* this
table; the closeout in Section 6 saves it.

Nothing here is a practice — it is assembly. Read the table when it prints and
check it against the printout from Section 2: same diseases, same quadrants, and
a reason on every row.

In [ ]:
import pandas as pd

# One row per transfer subject: the grid placement from Section 2, joined to the
# numbers from the gap table that justify it. The reason is the sentence you
# wrote; it travels with the row so the table is self-explaining.
rows = []
for d in transfer_diseases:
    p = placement[d]
    rows.append({
        "disease":         d,
        "within_host_acc": float(per_disease.loc[d, "within_host_acc"]),
        "cross_host_acc":  float(per_disease.loc[d, "cross_host_acc"]),
        "gap":             float(per_disease.loc[d, "gap"]),
        "n_cross_host":    int(per_disease.loc[d, "n_cross_host"]),
        "within_host_flag": per_disease.loc[d, "within_host_flag"],
        "transfer_row":    p["row"],        # held / collapsed / uninterpretable
        "pathogen":        p["pathogen"],   # same / different / unclear
        "quadrant":        p["quadrant"],
        "reason":          p["reason"],
    })

interpretation_table = pd.DataFrame(
    rows,
    columns=["disease", "within_host_acc", "cross_host_acc", "gap",
             "n_cross_host", "within_host_flag", "transfer_row", "pathogen",
             "quadrant", "reason"],
)

# Read it back before trusting it. Compare the disease/quadrant pairs here to
# the Section 2 printout — they should match exactly.
with pd.option_context("display.max_colwidth", 60):
    print(interpretation_table[["disease", "gap", "quadrant", "reason"]]
          .to_string(index=False))

# A small tally to eyeball the spread of outcomes.
print("\nQuadrant tally:")
for q, n in interpretation_table["quadrant"].value_counts().items():
    print(f"  {q:30s} {n}")

### 5) Frame the paper-level conclusion (per-disease claims, not sweeping ones — per Consideration 4)

The table is the evidence; the conclusion is what you claim from it. R2-Q4's
Consideration 4 is the rule that governs this section: commit to per-disease
claims your data supports, not one sweeping statement about transfer. A
classifier that transferred cleanly for one disease and collapsed for another
has not "transferred well" or "transferred poorly" — it did both, for different
reasons, and the paper says so disease by disease.

Write the `conclusion` field below. It is one short paragraph, in your own
words, and it should:

- **Name each disease's outcome by its grid cell**, not just its gap. "Late
  blight: host-template shortcut" says more than "Late blight: gap 0.31."
- **Carry the caveats you already raised.** A disease NB01 flagged near-chance
  (`within_host_flag == "near-chance"`, quadrant `unclear`) cannot anchor a
  transfer claim — say so. A disease placed `unclear` for pathogen identity is a
  measured-but-unexplained result — say that too, rather than guessing.
- **Honor the Section 3 constraint.** If you have a `clean_transfer` disease, the
  claim is "consistent with the model having learned disease features," scoped by
  the Noyan foreground caveat — not a bare "the model learned the disease."
- **Resist the sweeping sentence.** No "the classifier generalizes across hosts"
  line. The whole point of the per-disease grid is that such a sentence would be
  false for at least one disease.

This paragraph is the spine of your Week-4 paper. The cell below records it and
checks only that you have written it — the judgment is entirely yours.

In [ ]:
# EDIT the `conclusion` string. The structural facts are already in
# interpretation_table; this is the reading you commit to in the paper.
paper_conclusion = {
    "conclusion": (
        # One short paragraph. See the markdown above for what to cover:
        #  - each disease named by its grid cell, not just its gap;
        #  - near-chance and pathogen-unclear diseases carried with their caveat;
        #  - clean-transfer claims scoped by the Noyan foreground result;
        #  - no sweeping across-the-board sentence.
        "PLACEHOLDER — REWRITE BEFORE SUBMISSION."
    ),
}

# Semantic check: the conclusion must actually be written. This is a reminder,
# not a hard stop — the notebook still saves your work in Section 6 either way,
# the same soft-gate shape NB01 used for its within-host interpretation.
conclusion_text = paper_conclusion["conclusion"].strip()
if not conclusion_text or conclusion_text.startswith("PLACEHOLDER"):
    print("REMINDER: the paper conclusion is still the placeholder. Rewrite "
          "paper_conclusion['conclusion'] before you submit your final paper.")
else:
    n_words = len(conclusion_text.split())
    print(f"Conclusion recorded ({n_words} words). Read it once more against the "
          "interpretation table before Section 6.")

### 6) Closeout: save `interpretation_table.parquet`; submit final presentation and final paper

Week 4 is done once this notebook has saved the interpretation table and you have
submitted the two written deliverables. The table is the reproducible record of
what you concluded and why; the paper and presentation are where you say it in
your own voice.

In [ ]:
# Save the interpretation table — the one artifact this notebook produces. The
# paper conclusion is folded in as table-level metadata via the parquet schema's
# key-value store, so the reasoning travels with the numbers in a single file.
import json
import pyarrow as pa
import pyarrow.parquet as pq

interp_path = OUTPUT_DIR / "interpretation_table.parquet"

table = pa.Table.from_pandas(interpretation_table, preserve_index=False)
existing_meta = table.schema.metadata or {}
table = table.replace_schema_metadata({
    **existing_meta,
    b"paper_conclusion": paper_conclusion["conclusion"].strip().encode("utf-8"),
    b"diseases": json.dumps(diseases).encode("utf-8"),
})
pq.write_table(table, interp_path)

# Read it back and confirm it round-trips — the rows and the conclusion both.
reloaded = pq.read_table(interp_path)
assert reloaded.num_rows == len(interpretation_table), "Row count changed on save."
reloaded_meta = reloaded.schema.metadata or {}
assert reloaded_meta.get(b"paper_conclusion") is not None, (
    "Conclusion metadata didn't round-trip."
)
print(f"Wrote {interp_path}")
print(f"  {reloaded.num_rows} disease row(s); conclusion stored as table metadata.")

Two things to submit, both written by you, both built on the table you just
saved:

- **Your final presentation** — the Week-4 deliverable. Walk the 2×2 grid: which
  diseases landed where, and the one-line reason for each. Lead with the
  per-disease picture, not a single headline number.
- **Your final paper** — the per-disease conclusion from Section 5, with the
  table as its evidence and the Noyan (2022) caveat from Section 3 stated where
  it bears on your clean-transfer claims.

That closes R2-Q4. You started four weeks ago by pre-committing a disease list, a
hold-out scheme, and a reporting level before seeing any result; you trained a
within-host classifier and gated it on whether each disease was learnable at all;
you measured the cross-host gap disease by disease; and you turned each gap into
a claim about what the model learned and could not learn. The grid is the finding
— not that transfer "works" or "fails," but that it does different things for
different diseases, for reasons you can name.